# NEXUS Person Follower Robot — YOLOv8 Model Quantization to TFLite
## Quantization Pipeline: FP32 → INT8 & INT16 (16x8) TFLite

**Project**: NEXUS Person Follower Robot  
**Target Hardware**: Raspberry Pi 5 (Broadcom BCM2712 Quad-Core Cortex-A76 @ 2.4 GHz)  
**Domain**: Computer Vision / Mobile Robotics / Edge AI  

---

### Objective & Problem Resolution
Pada pengujian sebelumnya, model INT8 TFLite gagal mendeteksi orang (*0 detection / score 0*) karena:
1. **Kalibrasi Menggunakan Random Noise**: Generator kalibrasi sebelumnya memakai angka acak sintetik, sehingga parameter kuantisasi (*Scale* dan *Zero-Point*) hancur dan bobot layer terpotong (*clipped* ke nol).
2. **Kuantisasi Output Tensor ke INT8**: Menyetel `inference_output_type = tf.int8` memaksa output berupa bilangan bulat mentah `[-128, 127]`, sehingga logika decoding bounding box Ultralytics yang mengharapkan probabilitas float `[0.0, 1.0]` menjadi rusak.

**Perbaikan pada Notebook Ini**:
- Menggunakan **Real Images** (`coco8` / dataset orang asli) untuk kalibrasi aktivasi yang presisi.
- Mempertahankan antarmuka **Float32 pada Input dan Output** (`Quantize/Dequantize boundary`), sementara **seluruh komputasi internal berjalan dalam Full INT8**. Dengan cara ini, downstream pipeline (seperti `top_module.py` dan `person_detection.py`) dapat langsung membaca bounding box dan confidence score tanpa distorsi.
- Model TFLite langsung disimpan di folder utama **`models/yolo/`** sesuai permintaan.

## 1. Environment Setup & Dependency Verification

Verifikasi ketersediaan PyTorch, Ultralytics, dan TensorFlow.

In [10]:
import os
import sys
import platform
from pathlib import Path
import numpy as np
import cv2
import torch

print("=" * 65)
print("NEXUS QUANTIZATION PIPELINE — ENVIRONMENT AUDIT")
print("=" * 65)
print(f"OS Platform         : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python Version      : {sys.version.split()[0]}")
print(f"PyTorch Version     : {torch.__version__}")

try:
    import ultralytics
    from ultralytics import YOLO
    print(f"Ultralytics Version : {ultralytics.__version__}")
except ImportError:
    print("Ultralytics         : NOT INSTALLED (Run: pip install ultralytics)")

try:
    import tensorflow as tf
    print(f"TensorFlow Version  : {tf.__version__}")
except ImportError:
    print("TensorFlow          : NOT INSTALLED (Run: pip install tensorflow)")

print("=" * 65)


NEXUS QUANTIZATION PIPELINE — ENVIRONMENT AUDIT
OS Platform         : Linux 7.0.0-38-generic (x86_64)
Python Version      : 3.10.21
PyTorch Version     : 2.5.1
Ultralytics Version : 8.4.137
TensorFlow Version  : 2.15.1


## 2. Configuration & Model Path Definitions

Model `.tflite` hasil ekspor akan langsung disimpan di folder `models/yolo/`.

In [11]:
# Base directory paths
NOTEBOOK_DIR = Path.cwd().resolve() if Path.cwd().name == "yolo" else Path("models/yolo").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

# Target Base Model (Official YOLOv8 Nano checkpoint)
MODEL_PATH = NOTEBOOK_DIR / "yolov8n.pt"

# Output Directory: Langsung disimpan di models/yolo/
OUTPUT_DIR = NOTEBOOK_DIR

# Target File Names
INT8_TFLITE_PATH = OUTPUT_DIR / "nexus_person_detector_int8.tflite"
INT16_TFLITE_PATH = OUTPUT_DIR / "nexus_person_detector_int16.tflite"
FP16_TFLITE_PATH = OUTPUT_DIR / "nexus_person_detector_fp16.tflite"

# Input Image Dimension (640x640 per standard NEXUS spec)
IMG_SIZE = 640

print(f"Source Model Path   : {MODEL_PATH}")
print(f"Target Output Dir   : {OUTPUT_DIR}")
print(f"INT8 Target File    : {INT8_TFLITE_PATH}")
print(f"INT16 Target File   : {INT16_TFLITE_PATH}")
print(f"Resolution          : {IMG_SIZE}x{IMG_SIZE}")


Source Model Path   : /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n.pt
Target Output Dir   : /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo
INT8 Target File    : /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_int8.tflite
INT16 Target File   : /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_int16.tflite
Resolution          : 640x640


## 3. Real Image Representative Dataset Generator (Kalibrasi Presisi)

Untuk mencegah degradasi skor akurasi (yang membuat orang tidak terdeteksi), kalibrasi harus menggunakan **gambar pemandangan/manusia asli**, bukan angka acak sintetik.

Fungsi di bawah ini secara otomatis memindai gambar asli di `models/yolo/` (seperti foto COCO8 yang sudah ada) atau dataset lokal.

In [12]:
def find_real_calibration_images(max_images: int = 50):
    """Scans workspace for real natural photos for high-precision quantization calibration."""
    img_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        img_paths.extend(list(NOTEBOOK_DIR.rglob(ext)))
        if PROJECT_ROOT.exists():
            img_paths.extend(list((PROJECT_ROOT / "assets").rglob(ext)))
    # Filter out flowchart/diagram images
    real_photos = [p for p in img_paths if "flowchart" not in p.name.lower()]
    return real_photos[:max_images]

CALIB_IMAGES = find_real_calibration_images(max_images=50)
print(f"[Calibration] Ditemukan {len(CALIB_IMAGES)} foto asli untuk kalibrasi:")
for p in CALIB_IMAGES[:5]:
    print(f"  - {p.relative_to(NOTEBOOK_DIR) if p.is_relative_to(NOTEBOOK_DIR) else p.name}")

def representative_dataset_gen():
    """Generator providing real image tensors to calibrate dynamic activation ranges."""
    if len(CALIB_IMAGES) > 0:
        for p in CALIB_IMAGES:
            img = cv2.imread(str(p))
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype(np.float32) / 255.0
            yield [np.expand_dims(img, axis=0)]
    else:
        print("⚠️ PERINGATAN: Tidak ada foto asli ditemukan, fallback ke generator statik.")
        np.random.seed(42)
        for _ in range(50):
            yield [np.random.uniform(0.1, 0.9, size=(1, IMG_SIZE, IMG_SIZE, 3)).astype(np.float32)]

print("✅ Representative dataset generator siap.")


[Calibration] Ditemukan 8 foto asli untuk kalibrasi:
  - ~/PP2/data/coco8/images/train/000000000034.jpg
  - ~/PP2/data/coco8/images/train/000000000009.jpg
  - ~/PP2/data/coco8/images/train/000000000025.jpg
  - ~/PP2/data/coco8/images/train/000000000030.jpg
  - ~/PP2/data/coco8/images/val/000000000036.jpg
✅ Representative dataset generator siap.


## 4. Quantization to Full Integer INT8 (`nexus_person_detector_int8.tflite`)

### Kunci Perbaikan:
- **Komputasi Internal**: Dioptimasi penuh ke 8-bit integer (`INT8`).
- **Boundary Input & Output**: Dibiarkan tetap `Float32`. TFLite secara otomatis menyisipkan node Quantize di awal dan Dequantize di akhir. Ini memastikan probabilitas deteksi tetap `0.0 - 1.0` yang kompatibel 100% dengan YOLO decoder di `top_module.py`.

In [13]:
import shutil
import tensorflow as tf

# 1. Load official YOLOv8 model
model = YOLO(str(MODEL_PATH))
print(f"[Export] Model {MODEL_PATH.name} loaded successfully.")

# 2. Export SavedModel sebagai basis konversi
print("[Export] Mengekspor SavedModel...")
saved_model_path_str = model.export(format="saved_model", imgsz=IMG_SIZE)
saved_model_dir = Path(saved_model_path_str)
print(f"[Export] SavedModel siap di: {saved_model_dir}")

# 3. Konversi ke Full INT8 TFLite dengan Float32 I/O
print("[TFLite Converter] Mengonversi ke Full INT8 TFLite...")
converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Enforce integer quantization internal graph ops
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.TFLITE_BUILTINS]
# Input dan Output dibiarkan Float32 agar output probabilitas YOLO tetap akurat dan tidak korup

tflite_int8_model = converter.convert()

# Simpan langsung ke models/yolo/nexus_person_detector_int8.tflite
with open(INT8_TFLITE_PATH, "wb") as f:
    f.write(tflite_int8_model)

print(f"✅ INT8 Model Berhasil Disimpan: {INT8_TFLITE_PATH} ({INT8_TFLITE_PATH.stat().st_size / (1024*1024):.2f} MB)")


[Export] Model yolov8n.pt loaded successfully.
[Export] Mengekspor SavedModel...
Ultralytics 8.4.137 🚀 Python-3.10.21 torch-2.5.1 CPU (11th Gen Intel Core i3-1115G4 @ 3.00GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from '/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.4s, saved as '/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n.onnx' (12.3 MB)

TensorFlow SavedModel: starting export with tensorflow 2.15.1...
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640

Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 155, Total Ops 412, % non-converted = 37.62 %
 * 155 ARITH ops

- arith.constant:  155 occurrences  (f32: 131, i32: 24)



  (f32: 8)
  (f32: 17)
  (f32: 64)
  (f32: 58)
  (f32: 3)
  (f32: 59)
  (f32: 7)
  (f32: 8)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)
Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 155, Total Ops 543, % non-converted = 28.55 %
 * 155 ARITH ops

- arith.constant:  155 occurrences  (f16: 131, i32: 24)



  (f32: 8)
  (f32: 17)
  (f32: 64)
  (f32: 131)
  (f32: 58)
  (f32: 3)
  (f32: 59)
  (f32: 7)
  (f32: 8)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)


TensorFlow SavedModel: export success ✅ 18.6s, saved as '/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model' (30.7 MB)

Export complete (19.0s)
Results saved to /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model
Predict:         yolo predict task=detect model=/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model imgsz=640 
Validate:        yolo val task=detect model=/home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model imgsz=640 data=coco.yaml  
Visualize:       https://netron.app
[Export] SavedModel siap di: /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/yolov8n_saved_model
[TFLite Converter] Mengonversi ke Full INT8 TFLite...


Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 155, Total Ops 412, % non-converted = 37.62 %
 * 155 ARITH ops

- arith.constant:  155 occurrences  (f32: 131, i32: 24)



  (f32: 8)
  (f32: 17)
  (f32: 64)
  (f32: 58)
  (f32: 3)
  (f32: 59)
  (f32: 7)
  (f32: 8)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32


✅ INT8 Model Berhasil Disimpan: /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_int8.tflite (3.25 MB)


## 5. Quantization to 16-bit Integer (`nexus_person_detector_int16.tflite`)

### Kuantisasi 16x8 Integer:
- **Bobot**: 8-bit integer (`int8`)
- **Aktivasi**: 16-bit integer (`int16`)
- Menggunakan `EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8`.

In [14]:
print("[TFLite Converter] Mengonversi ke 16x8 Integer Quantized Model...")
converter_16 = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter_16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_16.representative_dataset = representative_dataset_gen

converter_16.target_spec.supported_ops = [
    tf.lite.OpsSet.EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS
]

tflite_int16_model = converter_16.convert()

# Simpan langsung ke models/yolo/nexus_person_detector_int16.tflite
with open(INT16_TFLITE_PATH, "wb") as f:
    f.write(tflite_int16_model)

print(f"✅ INT16 Model Berhasil Disimpan: {INT16_TFLITE_PATH} ({INT16_TFLITE_PATH.stat().st_size / (1024*1024):.2f} MB)")

# --- Bonus: Float16 (FP16) Half-Precision Export ---
print("[TFLite Converter] Mengonversi ke Float16 (FP16) Model...")
converter_fp16 = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]

tflite_fp16_model = converter_fp16.convert()
with open(FP16_TFLITE_PATH, "wb") as f:
    f.write(tflite_fp16_model)

print(f"✅ FP16 Model Berhasil Disimpan: {FP16_TFLITE_PATH} ({FP16_TFLITE_PATH.stat().st_size / (1024*1024):.2f} MB)")


[TFLite Converter] Mengonversi ke 16x8 Integer Quantized Model...


Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 155, Total Ops 412, % non-converted = 37.62 %
 * 155 ARITH ops

- arith.constant:  155 occurrences  (f32: 131, i32: 24)



  (f32: 8)
  (f32: 17)
  (f32: 64)
  (f32: 58)
  (f32: 3)
  (f32: 59)
  (f32: 7)
  (f32: 8)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)


✅ INT16 Model Berhasil Disimpan: /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_int16.tflite (3.35 MB)
[TFLite Converter] Mengonversi ke Float16 (FP16) Model...


Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 155, Total Ops 543, % non-converted = 28.55 %
 * 155 ARITH ops

- arith.constant:  155 occurrences  (f16: 131, i32: 24)



  (f32: 8)
  (f32: 17)
  (f32: 64)
  (f32: 131)
  (f32: 58)
  (f32: 3)
  (f32: 59)
  (f32: 7)
  (f32: 8)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)


✅ FP16 Model Berhasil Disimpan: /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_fp16.tflite (6.13 MB)


## 6. Model Footprint & Compression Analysis

Perbandingan ukuran file model hasil kuantisasi di folder `models/yolo/`:

In [15]:
models_to_compare = {
    "PyTorch Baseline (FP32)": MODEL_PATH,
    "ONNX (FP32)": NOTEBOOK_DIR / "yolov8n.onnx",
    "TFLite (FP16)": FP16_TFLITE_PATH,
    "TFLite (INT16 / 16x8)": INT16_TFLITE_PATH,
    "TFLite (INT8)": INT8_TFLITE_PATH
}

print("=" * 65)
print(f"{'Model Format / Precision':<26} | {'File Size (MB)':<15} | {'Path'}")
print("-" * 65)

for label, path in models_to_compare.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{label:<26} | {size_mb:>12.2f} MB | {path.name}")
    else:
        print(f"{label:<26} | {'N/A':>15} | Belum Diekspor")
print("=" * 65)


Model Format / Precision   | File Size (MB)  | Path
-----------------------------------------------------------------
PyTorch Baseline (FP32)    |         6.25 MB | yolov8n.pt
ONNX (FP32)                |        12.24 MB | yolov8n.onnx
TFLite (FP16)              |         6.13 MB | nexus_person_detector_fp16.tflite
TFLite (INT16 / 16x8)      |         3.35 MB | nexus_person_detector_int16.tflite
TFLite (INT8)              |         3.25 MB | nexus_person_detector_int8.tflite


## 7. Interactive Debugging & Verification Test (Deteksi Orang)

Di sel ini kita memvalidasi secara visual apakah model `nexus_person_detector_int8.tflite` yang baru dapat mendeteksi manusia dengan benar (membandingkan langsung hasil inferensi PyTorch `.pt` vs INT8 TFLite).

In [16]:
import matplotlib.pyplot as plt

# Pilih salah satu gambar tes yang berisi manusia
test_img_path = None
for p in CALIB_IMAGES:
    test_img_path = p
    break

if test_img_path is None or not test_img_path.exists():
    print("Gambar tes tidak ditemukan. Menggunakan gambar sample webcam / random.")
    test_frame = np.zeros((480, 640, 3), dtype=np.uint8)
else:
    test_frame = cv2.imread(str(test_img_path))
    print(f"[Testing] Menguji deteksi pada gambar: {test_img_path.name}")

# 1. Test dengan PyTorch Model (Baseline)
pt_results = model(test_frame, classes=[0], conf=0.30, verbose=False)[0]
pt_persons = len(pt_results.boxes) if pt_results.boxes is not None else 0
print(f"[PyTorch Baseline] Orang terdeteksi: {pt_persons}")
for b in pt_results.boxes:
    print(f"   - Conf: {float(b.conf[0]):.2f} | BBox: {[round(x) for x in b.xyxy[0].tolist()]}")

# 2. Test dengan INT8 TFLite Model yang baru dikuantisasi
int8_model = YOLO(str(INT8_TFLITE_PATH))
int8_results = int8_model(test_frame, classes=[0], conf=0.30, verbose=False)[0]
int8_persons = len(int8_results.boxes) if int8_results.boxes is not None else 0
print(f"\n[INT8 TFLite Baru] Orang terdeteksi: {int8_persons}")
for b in int8_results.boxes:
    print(f"   - Conf: {float(b.conf[0]):.2f} | BBox: {[round(x) for x in b.xyxy[0].tolist()]}")

if int8_persons > 0:
    print("\n🎉 VERIFIKASI BERHASIL! Model INT8 TFLite berhasil mendeteksi orang dengan akurat!")
else:
    print("\n⚠️ Catatan: Jika conf masih di bawah 0.50, coba jalankan dengan ambang batas conf=0.30.")


[Testing] Menguji deteksi pada gambar: 000000000034.jpg
[PyTorch Baseline] Orang terdeteksi: 0
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading /home/nexus_is/NEXUS_Person_Follower_Robot/models/yolo/nexus_person_detector_int8.tflite for LiteRT inference...

[INT8 TFLite Baru] Orang terdeteksi: 0

⚠️ Catatan: Jika conf masih di bawah 0.50, coba jalankan dengan ambang batas conf=0.30.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 8. Latency Benchmark (FPS & Throughput)

Mengukur kecepatan eksekusi model INT8 TFLite.

In [17]:
import time

def benchmark_tflite_model(model_path: Path, num_runs: int = 30):
    if not model_path.exists():
        return
    
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details['shape']
    dummy_input = np.random.uniform(0.0, 1.0, size=input_shape).astype(np.float32)
    
    # Warmup
    for _ in range(5):
        interpreter.set_tensor(input_details['index'], dummy_input)
        interpreter.invoke()
        
    # Benchmark
    times = []
    for _ in range(num_runs):
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details['index'], dummy_input)
        interpreter.invoke()
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)
        
    avg_lat = np.mean(times)
    fps = 1000.0 / avg_lat if avg_lat > 0 else 0.0
    print(f"[{model_path.name:<34}] Latensi: {avg_lat:5.2f} ms | Throughput: {fps:4.1f} FPS")

print("Menjalankan benchmark latensi...")
benchmark_tflite_model(INT8_TFLITE_PATH)
benchmark_tflite_model(INT16_TFLITE_PATH)
benchmark_tflite_model(FP16_TFLITE_PATH)


Menjalankan benchmark latensi...
[nexus_person_detector_int8.tflite ] Latensi: 89.69 ms | Throughput: 11.1 FPS
[nexus_person_detector_int16.tflite] Latensi: 1448.29 ms | Throughput:  0.7 FPS
[nexus_person_detector_fp16.tflite ] Latensi: 127.84 ms | Throughput:  7.8 FPS


## 9. Petunjuk Menjalankan di `top_module.py`

Karena file model sekarang langsung tersimpan di `models/yolo/`, kamu bisa langsung menjalankannya dengan perintah:

```bash
python top_module.py --model models/yolo/nexus_person_detector_int8.tflite --imgsz 640 --detect-interval 5
```